# Модуль с деревом решений

In [1]:
import pandas as pd

In [2]:
df_train = pd.read_csv("data\\train.csv", index_col='id')

Применяем те же преобразования, что и в show_data.ipynb и исключим признаки, которые имеют много категориальных значений:

In [3]:
from sklearn.preprocessing import FunctionTransformer

def preprocess(X):
    X = X.copy()

    X['job/study satisfaction'] = X['Job Satisfaction'].fillna(X['Study Satisfaction'])
    X['Work/Academic Pressure'] = X['Academic Pressure'].fillna(X['Work Pressure'])
    X.drop(columns=['Name', 'City', 'Job Satisfaction', 'Study Satisfaction', 'Academic Pressure', 'Work Pressure'], inplace=True)
    X.fillna({'Profession': 'unemployed'}, inplace=True)
    X.drop('CGPA', inplace=True, axis=1)

    return X

prep = FunctionTransformer(preprocess, validate=False)

Применяем OneHot энкодинг для категориальных признаков

In [4]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector

encoder = ColumnTransformer([
    ("onehot", OneHotEncoder(sparse_output=False), make_column_selector(dtype_include=object))
])

Оценивать качество будем при помощи кросс-валидации

In [5]:
from sklearn.model_selection import cross_val_score

def cv(model, X, y):
    score = cross_val_score(
        model,
        X,
        y,
        scoring='f1',
    )

    return score.mean()

In [6]:
X = df_train.iloc[:, :-1]
y = df_train.iloc[:, -1]

Строим пайплайн и оцениваем качество

In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('prep', prep),
    ('encode', encoder),
    ('model', DecisionTreeClassifier()),
])

cv(pipe, X, y)

np.float64(0.43613597490718925)

качество получилось довольно низким, попробуем воспользоваться GridSearchCV для перебора гиперпараметров

In [8]:
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([
    ('prep', prep),
    ('encode', encoder),
    ('model', DecisionTreeClassifier()),
])

param_grid = {
    "model__criterion": ["gini", "entropy", "log_loss"],
    "model__max_depth": [3, 5, 7, 10, None],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__splitter": ["best", "random"]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    refit=True
)

grid.fit(X, y)
grid.best_score_

np.float64(0.5736306272671721)

Показатель f1 score заметно вырос, дополнительно попробуем кодировать бинарные признаки при помощи label encoding, чтобы уменьшить количество признаков 

In [9]:
from sklearn.preprocessing import OrdinalEncoder

encoder = ColumnTransformer([
        ("onehot", OneHotEncoder(sparse_output=False), ['Profession', 'Sleep Duration', 'Dietary Habits', 'Degree']),
        ("label", OrdinalEncoder(), ['Gender', 'Working Professional or Student', 'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']),
])

pipe = Pipeline([
    ("prep", prep),
    ("encode", encoder),
    ("model", DecisionTreeClassifier()),
])

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    refit=True
)

grid.fit(X, y)
grid.best_score_

np.float64(0.5735736509586713)